# Notebook 01: Training NHITS on French Bakery Data

## Learning Objectives

By the end of this notebook you will be able to:

1. Load time series data into the Nixtla long format (`unique_id`, `ds`, `y`)
2. Train an NHITS model using NeuralForecast
3. Evaluate forecast quality with MAE and MSE
4. Plot actual vs predicted forecasts

**Estimated time**: 12–15 minutes (including ~5 min training)

**Prerequisites**: Guide 01 (Neural Architectures)

---

## Setup

Install required packages if not already present. NeuralForecast requires PyTorch — training on CPU is fine for this notebook.

In [ ]:
# Install dependencies (skip if already installed)
# !pip install neuralforecast utilsforecast matplotlib pandas --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("Imports complete")

## 1. Load the French Bakery Dataset

The French Bakery dataset contains daily sales counts for multiple bakery products from a real boulangerie in France. The dataset is hosted by Nixtla in their transfer-learning repository.

NeuralForecast requires data in the **long format** with exactly three columns:

- `unique_id`: identifies the time series (one per product)
- `ds`: the date (datetime64)
- `y`: the target variable (sales count)

This format is also called the "Nixtla format" or "tidy" format.

In [ ]:
# Load the French Bakery dataset
# Source: Nixtla transfer-learning-time-series repository
# Original data: Kaggle — French Bakery Daily Sales (matthieugimbert)
url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series"
    "/main/datasets/french_bakery.csv"
)
df = pd.read_csv(url, parse_dates=["ds"])

# Inspect the data
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nProducts (unique_id):", df["unique_id"].unique().tolist())
print("\nDate range:", df["ds"].min().date(), "to", df["ds"].max().date())
print("\nFirst 5 rows:")
df.head()

## 2. Explore the Data

Before training, visualize the sales patterns. This step confirms:
- The data has weekly seasonality (important for setting `input_size`)
- There are no major gaps or data quality issues
- Sales volumes differ across products (the model must handle multiple scales)

In [ ]:
# Plot daily sales for each product
# This reveals the weekly pattern, overall trend, and outliers
products = df["unique_id"].unique()
n_products = len(products)

fig, axes = plt.subplots(n_products, 1, figsize=(14, 3 * n_products), sharex=True)
if n_products == 1:
    axes = [axes]

for ax, product in zip(axes, products):
    series = df[df["unique_id"] == product].sort_values("ds")
    ax.plot(series["ds"], series["y"], linewidth=0.8, color="steelblue", alpha=0.8)
    ax.set_ylabel("Daily Sales", fontsize=10)
    ax.set_title(product, fontsize=11, fontweight="bold")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

axes[-1].set_xlabel("Date")
fig.suptitle("French Bakery — Daily Sales by Product", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Summary statistics per product
summary = df.groupby("unique_id")["y"].agg(["mean", "std", "min", "max"]).round(1)
print("\nSummary statistics:")
summary

## 3. Prepare the Training and Test Sets

We hold out the last 7 days (one forecast horizon) of data as the test set. The model trains on everything before the cutoff date.

This is a single train/test split. In Notebook 02 we use rolling cross-validation for a more robust evaluation.

In [ ]:
# Split the last H=7 days as the test set
# The model will train on df_train and we compare predictions against df_test
H = 7  # forecast horizon (7 days)

# Find the cutoff date: H days before the end of the series
max_date = df["ds"].max()
cutoff = max_date - pd.Timedelta(days=H)

df_train = df[df["ds"] <= cutoff].copy()
df_test = df[df["ds"] > cutoff].copy()

print(f"Forecast horizon H = {H} days")
print(f"Training cutoff: {cutoff.date()}")
print(f"Training rows: {len(df_train)}")
print(f"Test rows: {len(df_test)}")
print(f"\nTest dates: {df_test['ds'].min().date()} to {df_test['ds'].max().date()}")

## 4. Train NHITS

We configure NHITS with:
- `h=7`: predict 7 days ahead
- `input_size=28`: use 28 days of history (4× horizon — four complete weekly cycles)
- `max_steps=500`: fast training for exploration (use 1000+ for production)
- `scaler_type="robust"`: normalize using median and IQR, robust to sales spikes

Training time on CPU: approximately 3–5 minutes.

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS

# Configure the model
# input_size=28: four complete weekly cycles of history
# scaler_type='robust': handles bakery sales outliers (holiday spikes)
# max_steps=500: sufficient for demonstration; use 1000+ for production
model = NHITS(
    h=H,
    input_size=28,
    max_steps=500,
    scaler_type="robust",
)

# NeuralForecast wraps models and handles the train loop
nf = NeuralForecast(models=[model], freq="D")

print("Training NHITS on French Bakery data...")
print(f"Series: {df_train['unique_id'].nunique()} products")
print(f"Training observations per series: ~{len(df_train) // df_train['unique_id'].nunique()}")
print()

# Fit on the training data
# NeuralForecast creates sliding windows internally — no manual windowing needed
nf.fit(df_train)

print("Training complete.")

## 5. Generate Forecasts

`nf.predict()` generates forecasts for the next `h=7` days after the last observation in the training data. The output is a DataFrame with one row per (product, forecast date).

In [ ]:
# Generate 7-day ahead forecasts
# predict() automatically uses the last input_size observations from training
forecast = nf.predict()

print("Forecast output:")
print(forecast.head(14))  # show two products
print(f"\nForecast shape: {forecast.shape}")
print(f"Columns: {forecast.columns.tolist()}")

## 6. Evaluate: MAE and MSE

We compare the forecasts to the actual test values using:

- **MAE** (Mean Absolute Error): average absolute difference — same units as sales counts
- **MSE** (Mean Squared Error): penalizes large errors more heavily

Lower is better for both metrics.

In [ ]:
from utilsforecast.losses import mae, mse

# Merge forecasts with actuals for evaluation
# Both DataFrames have unique_id and ds columns — merge on those
eval_df = forecast.merge(df_test[["unique_id", "ds", "y"]], on=["unique_id", "ds"])

# Compute MAE and MSE per product
# utilsforecast's mae/mse functions compute element-wise, so we aggregate
results = []
for product in eval_df["unique_id"].unique():
    prod_df = eval_df[eval_df["unique_id"] == product]
    product_mae = mae(prod_df["y"], prod_df["NHITS"]).mean()
    product_mse = mse(prod_df["y"], prod_df["NHITS"]).mean()
    results.append({
        "product": product,
        "MAE": round(product_mae, 2),
        "MSE": round(product_mse, 2),
        "RMSE": round(product_mse ** 0.5, 2),
    })

results_df = pd.DataFrame(results).set_index("product")

# Overall (across all products)
overall_mae = mae(eval_df["y"], eval_df["NHITS"]).mean()
overall_mse = mse(eval_df["y"], eval_df["NHITS"]).mean()

print("Per-product metrics:")
print(results_df)
print(f"\nOverall MAE:  {overall_mae:.2f}")
print(f"Overall MSE:  {overall_mse:.2f}")
print(f"Overall RMSE: {overall_mse**0.5:.2f}")

## 7. Plot Actual vs Predicted

Visualizing the forecast alongside the actual values and the training history is the most important diagnostic. Numbers alone (MAE=16) are hard to interpret — seeing the plot tells you whether errors are systematic (always too low, missing peaks) or random.

In [ ]:
# Plot actual vs predicted for each product
# Show the last 60 days of training + the 7-day forecast
context_days = 60  # how many training days to show for context

fig, axes = plt.subplots(n_products, 1, figsize=(14, 4 * n_products))
if n_products == 1:
    axes = [axes]

for ax, product in zip(axes, products):
    # Training history (last 60 days)
    train_series = df_train[df_train["unique_id"] == product].sort_values("ds")
    train_tail = train_series.tail(context_days)
    
    # Test actuals and forecasts
    test_series = df_test[df_test["unique_id"] == product].sort_values("ds")
    pred_series = forecast[forecast["unique_id"] == product].sort_values("ds")
    
    # Plot training history
    ax.plot(
        train_tail["ds"], train_tail["y"],
        color="steelblue", linewidth=1.2, label="Actual (train)"
    )
    # Plot test actuals
    ax.plot(
        test_series["ds"], test_series["y"],
        color="black", linewidth=1.5, linestyle="-", marker="o",
        markersize=4, label="Actual (test)"
    )
    # Plot NHITS forecast
    ax.plot(
        pred_series["ds"], pred_series["NHITS"],
        color="tomato", linewidth=1.8, linestyle="--", marker="s",
        markersize=4, label="NHITS forecast"
    )
    
    # Vertical line at the train/test cutoff
    ax.axvline(cutoff, color="gray", linestyle=":", linewidth=1.5, label="Cutoff")
    
    # Metrics annotation
    product_mae = results_df.loc[product, "MAE"]
    ax.set_title(f"{product}  |  MAE = {product_mae:.1f}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Daily Sales")
    ax.legend(loc="upper left", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

axes[-1].set_xlabel("Date")
fig.suptitle("NHITS Forecast vs Actual — French Bakery (h=7 days)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Error Analysis

A good forecast not only has low average error, but also has errors that are **random** (not systematically biased). The residual plot below helps detect:

- **Bias**: errors consistently positive or negative → model is systematically over/under-predicting
- **Day-of-week pattern**: errors larger on specific days → model misses a seasonal pattern
- **Outliers**: a single extremely large error → likely a holiday not captured by the model

In [ ]:
# Compute residuals: actual - predicted
# Positive residual: model under-predicted
# Negative residual: model over-predicted
eval_df["residual"] = eval_df["y"] - eval_df["NHITS"]
eval_df["day_of_week"] = eval_df["ds"].dt.day_name()

# Plot residuals per product
fig, axes = plt.subplots(1, n_products, figsize=(5 * n_products, 4), sharey=False)
if n_products == 1:
    axes = [axes]

for ax, product in zip(axes, products):
    prod_res = eval_df[eval_df["unique_id"] == product].sort_values("ds")
    
    # Bar chart of residuals, colored by sign
    colors = ["tomato" if r < 0 else "steelblue" for r in prod_res["residual"]]
    ax.bar(
        range(len(prod_res)), prod_res["residual"],
        color=colors, alpha=0.8, edgecolor="black", linewidth=0.5
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xticks(range(len(prod_res)))
    ax.set_xticklabels(
        [d.strftime("%a\n%b%d") for d in prod_res["ds"]], fontsize=8
    )
    ax.set_title(product, fontsize=11, fontweight="bold")
    ax.set_ylabel("Residual (actual - predicted)")

fig.suptitle("Forecast Residuals by Day", fontsize=13)
plt.tight_layout()
plt.show()

# Bias check: mean residual close to zero means no systematic bias
mean_residual = eval_df.groupby("unique_id")["residual"].mean().round(2)
print("Mean residual per product (should be near zero if unbiased):")
print(mean_residual)

## 9. Summary

In this notebook you:

1. Loaded the French Bakery dataset into the Nixtla long format
2. Trained NHITS with `h=7`, `input_size=28`, `scaler_type="robust"`
3. Generated a 7-day point forecast for each product
4. Computed MAE and MSE per product and overall
5. Visualized actual vs predicted and analyzed residuals

### What the Numbers Mean

A MAE of ~16 units means the model's typical prediction is about 16 baguettes off from the actual daily sales. In the context of mean sales around 180 units/day, this is approximately **9% relative error** — reasonable for a 7-day ahead forecast.

### Next Steps

- **Notebook 02**: Use `cross_validation()` to evaluate across multiple windows and compare NHITS against a DLinear baseline
- **Guide 02**: Tune `input_size`, `scaler_type`, and loss functions to improve these numbers
- **Exercise 01**: Try different `input_size` values and compare MAE